# FinTrust LLM forecaster, experiment 3 only (Phi-3-medium)

Rerun of experiment 3 in a fresh session, after the combined run ran out of GPU memory on this model.
Same setup: GPU T4 x2, internet on, `HF_TOKEN` secret, `nse_dataset` attached. Phi-3-medium needs no licence.
Run with *Save & Run All (Commit)*; roughly 2 to 4 hours (estimate). Download `llm_outputs_exp3.zip` from the Output tab.

In [ ]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import os, sys, glob, subprocess, hashlib, shutil, time
REPO = 'https://github.com/IAbdullahSlash/fintrust.git'
BRANCH = 'claude/project-thread-k1tvaj'          # falls back to the default branch once this one is merged
if not os.path.exists('fintrust'):
    if subprocess.call(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, 'fintrust']) != 0:
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO, 'fintrust'])
print(subprocess.check_output(['git', '-C', 'fintrust', 'log', '-1', '--format=%H %s']).decode())

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate yfinance bitsandbytes
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Price data: the paper's copy if attached as a Kaggle Dataset, otherwise a fresh Yahoo Finance download
found = glob.glob('/kaggle/input/**/nse_daily.csv', recursive=True)
if found: PRICES = found[0]
else:
    subprocess.check_call([sys.executable, 'fintrust/data/download_nse.py']); PRICES = 'fintrust/data/raw/nse_daily.csv'
print(PRICES, hashlib.sha256(open(PRICES, 'rb').read()).hexdigest())

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login; login(token=os.environ['HF_TOKEN'])
sys.path.insert(0, 'fintrust/paper/experiments/llm')
import llm_forecaster as L
REQ, OUT = '/kaggle/working/requests', '/kaggle/working/outputs'
counts = L.build(PRICES, REQ, label_csv='fintrust/paper/experiments/fair/nse_vol_forecasts.csv')

In [ ]:
# Experiment 3: Phi-3-medium-4k-instruct (cutoff Oct 2023), 8-bit weights, experiment 1 prompts.
# Layers are spread over both GPUs with headroom, and a batch that runs out of memory is retried in halves.
import gc
be = L.HFBackend('microsoft/Phi-3-medium-4k-instruct', max_batch=8, quant='8bit', gpu_frac=0.8); print('revision', be.revision)
for i in range(torch.cuda.device_count()): print(i, round(torch.cuda.memory_allocated(i) / 2**30, 1), 'GiB used')
t0 = time.time(); p = L.run(be, f'{REQ}/clean.jsonl', OUT, 'phi3_medium', limit=16); print(f'{time.time() - t0:.1f}s for 16'); print(open(p).read()[:1200])

In [ ]:
for cond in ['clean', 'news_alarm', 'news_calm', 'persona', 'clean_pre', 'named', 'named_pre']:
    t0 = time.time(); L.run(be, f'{REQ}/{cond}.jsonl', OUT, 'phi3_medium'); print(cond, f'{(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
import json
for f in sorted(glob.glob(f'{OUT}/*.jsonl')):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), len(rows), 'parse failures:', sum(not r['parse_ok'] for r in rows), 'mean p: %.3f' % (sum(r['p'] for r in rows) / len(rows)))
shutil.make_archive('/kaggle/working/llm_outputs_exp3', 'zip', OUT); print('download llm_outputs_exp3.zip from the Output tab')